# LeetCode 806 — Number of Lines To Write String bằng Python

**Link đề:** https://leetcode.com/problems/number-of-lines-to-write-string/
**Độ khó:** Easy · **Chủ đề:** Array, String, Simulation (mô phỏng)

```python
class Solution:
    def numberOfLines(self, widths: List[int], s: str) -> List[int]:
        ...
```

## Mục tiêu học
1. Dùng mảng `widths` như **bảng tra** độ rộng theo chữ cái: `widths[ord(ch) - ord('a')]`.
2. Hiểu thuật toán **tham lam** (greedy) "xếp được thì xếp, không vừa thì xuống dòng".
3. Viết mô phỏng dễ hiểu (dựng từng dòng thật) → rút gọn chỉ còn **hai biến đếm** `lines`, `width`.
4. Thấy vì sao công thức "tổng độ rộng chia 100" là **sai** (chữ không được cắt đôi).
5. Trace, chứng minh, Time/Space, luyện biến thể (word wrap, giới hạn khác 100).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho chuỗi `s` gồm chữ thường và mảng `widths` dài 26: `widths[0]` là độ rộng (pixel) của `'a'`, `widths[1]` của `'b'`, ...

Viết `s` lên nhiều dòng, **mỗi dòng không quá 100 pixel**. Bắt đầu từ đầu `s`, viết **nhiều chữ nhất có thể** lên dòng 1 sao cho tổng độ rộng ≤ 100; rồi từ chỗ dừng, tiếp tục ở dòng 2, ... cho tới hết `s`.

Trả về `[tổng số dòng, độ rộng dòng cuối]`.

**Ví dụ 1:** mọi chữ rộng 10, `s = "abcdefghijklmnopqrstuvwxyz"` → `[3, 60]`
```text
abcdefghij  // 100 px
klmnopqrst  // 100 px
uvwxyz      // 60 px
```
**Ví dụ 2:** `widths[0] = 4` (chữ `a`), các chữ khác 10, `s = "bbbcccdddaaa"` → `[2, 4]`
```text
bbbcccdddaa  // 98 px  (9 chữ × 10 + 2 chữ a × 4)
a            // 4 px   (thêm 'a' nữa sẽ thành 102 > 100)
```

**Constraints:** `widths.length == 26`; `2 <= widths[i] <= 10`; `1 <= s.length <= 1000`; `s` chỉ có chữ thường.

In [1]:
from typing import List      # LeetCode dùng List[int] trong chữ ký; cần import khi chạy ngoài LeetCode

widths1 = [10] * 26
s1 = "abcdefghijklmnopqrstuvwxyz"
widths2 = [4] + [10] * 25
s2 = "bbbcccdddaaa"
print(len(widths1), widths2[:3], "...")

26 [4, 10, 10] ...


## 2. Nền tảng Python

### 2.1. `[10] * 26` và `[4] + [10] * 25`
- `[x] * k` tạo list gồm `k` phần tử `x`.
- `list1 + list2` nối hai list.

### 2.2. `typing.List`
`List[int]` chỉ là **chú thích kiểu** (type hint): "list các số nguyên". Python không kiểm tra khi chạy; viết `list[int]` (Python ≥ 3.9) cũng tương đương.

### 2.3. Tra độ rộng của một chữ
`ord(ch) - ord('a')` đổi `'a'..'z'` thành `0..25` → `widths[...]`.

In [2]:
print([10] * 3, [4] + [10] * 2)
for ch in "abz":
    print(ch, "-> chỉ số", ord(ch) - ord("a"), "-> rộng", widths2[ord(ch) - ord("a")])

[10, 10, 10] [4, 10, 10]
a -> chỉ số 0 -> rộng 4
b -> chỉ số 1 -> rộng 10
z -> chỉ số 25 -> rộng 10


### 2.4. Type hint không được kiểm tra khi chạy

Cell dưới cho thấy hàm khai báo `-> List[int]` vẫn chạy khi trả về chuỗi — type hint chỉ giúp người đọc và công cụ kiểm tra tĩnh (mypy, IDE). Vì vậy ta phải **tự** bảo đảm trả về đúng kiểu `list` `[lines, width]`.

In [3]:
def demo_hint(x: int) -> List[int]:
    return "không phải list"
print(demo_hint(5), type(demo_hint(5)))

không phải list <class 'str'>


## 3. Nền tảng tư duy: tham lam (greedy) — đúng như đề mô tả

Đề **đã quy định** cách xếp: mỗi dòng nhét nhiều chữ nhất có thể, theo thứ tự. Không có lựa chọn nào khác → ta chỉ cần **mô phỏng** chính xác.

Với mỗi chữ `ch` rộng `w`:
- Nếu `width_hiện_tại + w <= 100` → đặt vào dòng hiện tại.
- Ngược lại → **xuống dòng mới**, chữ `ch` là chữ đầu dòng mới (`width = w`).

Chú ý dấu `<=`: dòng **đúng bằng 100** vẫn hợp lệ (ví dụ 1, dòng 1 = 100 px).

In [4]:
cur, w = 98, 4
print("98 + 4 =", cur + w, "<= 100 ?", cur + w <= 100, "-> phải xuống dòng")
cur, w = 90, 10
print("90 + 10 =", cur + w, "<= 100 ?", cur + w <= 100, "-> vẫn vừa")

98 + 4 = 102 <= 100 ? False -> phải xuống dòng
90 + 10 = 100 <= 100 ? True -> vẫn vừa


## 4. Cách 1 — Mô phỏng trực quan: dựng từng dòng thật

Lưu danh sách các dòng (chuỗi) và độ rộng từng dòng. Dễ kiểm chứng bằng mắt, nhưng tốn bộ nhớ lưu văn bản.

In [5]:
def layout_lines(widths: list[int], s: str):
    lines = [""]          # bắt đầu với một dòng rỗng
    line_widths = [0]
    for ch in s:
        w = widths[ord(ch) - ord("a")]
        if line_widths[-1] + w > 100:     # không vừa -> mở dòng mới
            lines.append("")
            line_widths.append(0)
        lines[-1] += ch                   # thêm chữ vào dòng cuối
        line_widths[-1] += w
    return lines, line_widths

def number_of_lines_layout(widths, s):
    lines, line_widths = layout_lines(widths, s)
    return [len(lines), line_widths[-1]]

for ws, s in [(widths1, s1), (widths2, s2)]:
    lines, lw = layout_lines(ws, s)
    for text, w in zip(lines, lw):
        print(f"{text:<12} // {w} px")
    print("=>", number_of_lines_layout(ws, s))
    print()

abcdefghij   // 100 px
klmnopqrst   // 100 px
uvwxyz       // 60 px
=> [3, 60]

bbbcccdddaa  // 98 px
a            // 4 px
=> [2, 4]



### 4.1. `zip` — duyệt song song hai list

Cell trên dùng `for text, w in zip(lines, lw)`: `zip` ghép phần tử cùng vị trí của hai list thành cặp `(lines[0], lw[0]), (lines[1], lw[1]), ...`, dừng khi list ngắn hơn hết.

In [6]:
print(list(zip(["abc", "de"], [30, 20])))
for name, score in zip(["An", "Bình", "Chi"], [8, 9]):
    print(name, score)          # "Chi" bị bỏ vì list điểm ngắn hơn

[('abc', 30), ('de', 20)]
An 8
Bình 9


`lines[-1] += ch` sửa **phần tử cuối** của list: lấy chuỗi cũ, nối thêm, gán lại vào vị trí `-1`.

**Độ phức tạp cách 1:** `n = len(s)` → `O(n)` thời gian; bộ nhớ `O(n)` vì lưu toàn bộ văn bản theo dòng.

## 5. Nút thắt và insight — chỉ cần **2 con số**

Đề chỉ hỏi **số dòng** và **độ rộng dòng cuối**. Ta không cần nhớ nội dung các dòng trước: khi xuống dòng, dòng cũ "đã xong", không bao giờ quay lại.

→ Trạng thái chỉ là `(lines, width)`:
- `lines`: số dòng đã mở (bắt đầu `1` — vì `s` không rỗng, luôn có ít nhất một dòng).
- `width`: độ rộng dòng hiện tại (bắt đầu `0`).

Bộ nhớ giảm từ `O(n)` xuống `O(1)`.

In [7]:
import sys
long_s = "abcdefghijklmnopqrstuvwxyz" * 38          # 988 chữ, gần giới hạn 1000
lines_txt, lw = layout_lines(widths1, long_s)
mem_layout = sum(sys.getsizeof(t) for t in lines_txt) + sys.getsizeof(lines_txt) + sys.getsizeof(lw)
mem_counters = sys.getsizeof(1) + sys.getsizeof(0)
print("số dòng:", len(lines_txt))
print("bộ nhớ (ước lượng) cách dựng dòng :", mem_layout, "byte")
print("bộ nhớ hai biến đếm               :", mem_counters, "byte")

số dòng: 99
bộ nhớ (ước lượng) cách dựng dòng : 6887 byte
bộ nhớ hai biến đếm               : 56 byte


`sys.getsizeof(obj)` trả kích thước (byte) của một object — chỉ dùng để **ước lượng**; con số cho thấy bộ nhớ của cách dựng dòng tăng theo độ dài `s`, còn hai biến đếm thì cố định.

## 6. Lời giải chính — hai biến đếm

```python
lines, width = 1, 0                      # (1) dòng đầu tiên, đang rỗng
for ch in s:
    w = widths[ord(ch) - ord('a')]       # (2) tra độ rộng
    if width + w > 100:                  # (3) không vừa?
        lines += 1                       #     mở dòng mới
        width = w                        #     ch là chữ đầu dòng mới
    else:
        width += w                       # (4) vừa: đặt tiếp vào dòng
return [lines, width]
```
- `(3)` dùng `>` 100 (tương đương "không `<=` 100").
- Khi xuống dòng, `width = w` chứ **không** phải `0` — chữ `ch` vẫn phải được viết!

In [8]:
def number_of_lines(widths: list[int], s: str) -> list[int]:
    lines, width = 1, 0
    for ch in s:
        w = widths[ord(ch) - ord("a")]
        if width + w > 100:
            lines += 1
            width = w
        else:
            width += w
    return [lines, width]

print(number_of_lines(widths1, s1), number_of_lines(widths2, s2))

[3, 60] [2, 4]


## 7. TRACE từng bước

Ví dụ 2 (`a` rộng 4, còn lại 10), `s = "bbbcccdddaaa"`:

| i | ch | w | width trước | width + w | > 100? | lines sau | width sau |
|---|---|---|---|---|---|---|---|
| 0 | b | 10 | 0 | 10 | không | 1 | 10 |
| ... | ... | ... | ... | ... | ... | ... | ... |
| 8 | d | 10 | 80 | 90 | không | 1 | 90 |
| 9 | a | 4 | 90 | 94 | không | 1 | 94 |
| 10 | a | 4 | 94 | 98 | không | 1 | 98 |
| 11 | a | 4 | 98 | 102 | **có** | 2 | 4 |

In [9]:
def trace_number_of_lines(widths, s):
    lines, width = 1, 0
    print(" i | ch |  w | width trước | width+w | xuống dòng? | lines | width sau")
    for i, ch in enumerate(s):
        w = widths[ord(ch) - ord("a")]
        before = width
        new_line = width + w > 100
        if new_line:
            lines += 1
            width = w
        else:
            width += w
        print(f"{i:>2} | {ch:>2} | {w:>2} | {before:>11} | {before + w:>7} | {str(new_line):>11} | {lines:>5} | {width:>9}")
    print("Kết quả:", [lines, width])
    return [lines, width]

_ = trace_number_of_lines(widths2, s2)

 i | ch |  w | width trước | width+w | xuống dòng? | lines | width sau
 0 |  b | 10 |           0 |      10 |       False |     1 |        10
 1 |  b | 10 |          10 |      20 |       False |     1 |        20
 2 |  b | 10 |          20 |      30 |       False |     1 |        30
 3 |  c | 10 |          30 |      40 |       False |     1 |        40
 4 |  c | 10 |          40 |      50 |       False |     1 |        50
 5 |  c | 10 |          50 |      60 |       False |     1 |        60
 6 |  d | 10 |          60 |      70 |       False |     1 |        70
 7 |  d | 10 |          70 |      80 |       False |     1 |        80
 8 |  d | 10 |          80 |      90 |       False |     1 |        90
 9 |  a |  4 |          90 |      94 |       False |     1 |        94
10 |  a |  4 |          94 |      98 |       False |     1 |        98
11 |  a |  4 |          98 |     102 |        True |     2 |         4
Kết quả: [2, 4]


In [10]:
# Trace ví dụ 1, chỉ in những bước xuống dòng để thấy ranh giới 100 px
lines, width = 1, 0
for i, ch in enumerate(s1):
    w = widths1[ord(ch) - 97]
    if width + w > 100:
        print(f"i={i} ch={ch!r}: {width} + {w} = {width + w} > 100 -> dòng {lines + 1}")
        lines, width = lines + 1, w
    else:
        width += w
print("Kết quả:", [lines, width])

i=10 ch='k': 100 + 10 = 110 > 100 -> dòng 2
i=20 ch='u': 100 + 10 = 110 > 100 -> dòng 3
Kết quả: [3, 60]


## 8. Code nộp LeetCode — lời giải chính

In [11]:
class Solution:
    def numberOfLines(self, widths: List[int], s: str) -> List[int]:
        lines, width = 1, 0
        for ch in s:
            w = widths[ord(ch) - ord("a")]
            if width + w > 100:
                lines += 1
                width = w
            else:
                width += w
        return [lines, width]

sol = Solution()
assert sol.numberOfLines(widths1, s1) == [3, 60]
assert sol.numberOfLines(widths2, s2) == [2, 4]
print("Solution OK")

Solution OK


## 9. Cách 3 — Biến thể cài đặt: "đặt trước, kiểm sau"

Một cách viết khác cũng hay gặp: cộng `w` vào `width` **trước**, nếu vượt 100 thì mở dòng mới với `width = w`. Logic tương đương, giúp luyện mắt đọc code khác dạng.

In [12]:
class Solution:
    def numberOfLines(self, widths: List[int], s: str) -> List[int]:
        lines, width = 1, 0
        for ch in s:
            w = widths[ord(ch) - 97]
            width += w
            if width > 100:          # vừa đặt thì tràn -> chữ này sang dòng mới
                lines += 1
                width = w
        return [lines, width]

print(Solution().numberOfLines(widths1, s1), Solution().numberOfLines(widths2, s2))

[3, 60] [2, 4]


## 10. Cách 4 — Dùng `functools.reduce` (phong cách hàm)

Trạng thái `(lines, width)` được "gấp" (fold) qua từng chữ. `reduce(f, iterable, init)` gọi `f(state, x)` lần lượt và trả về state cuối. Ngắn gọn nhưng khó đọc hơn với người mới — chỉ để biết.

In [13]:
from functools import reduce

print(reduce(lambda acc, x: acc + x, [1, 2, 3, 4], 0))    # ((((0+1)+2)+3)+4) = 10

def step(state, w):
    lines, width = state
    return (lines + 1, w) if width + w > 100 else (lines, width + w)

def number_of_lines_reduce(widths, s):
    return list(reduce(step, (widths[ord(c) - 97] for c in s), (1, 0)))

print(number_of_lines_reduce(widths1, s1), number_of_lines_reduce(widths2, s2))

10
[3, 60] [2, 4]


## 11. Vì sao đúng?

**Invariant:** sau khi xử lý `k` chữ đầu, `lines` = số dòng mà quy trình của đề dùng cho `s[:k]`, và `width` = độ rộng dòng cuối của bố cục đó.

- `k = 1`: chữ đầu luôn vừa (`w <= 10 <= 100`) → `lines = 1, width = w`. ✔
- Bước `k → k+1`: quy trình của đề đặt chữ mới vào dòng hiện tại **nếu** vẫn ≤ 100, ngược lại mở dòng mới — đúng y hệt nhánh `if/else`. Các dòng trước không thay đổi nên không cần nhớ.
- Kết thúc: `[lines, width]` là đáp án.

Không bao giờ có "một chữ không vừa cả dòng trống" vì `widths[i] <= 10 < 100`.

## 12. Độ phức tạp và so sánh

`n = len(s)` (≤ 1000).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Dựng dòng thật | `O(n)` | `O(n)` | Dễ kiểm chứng bằng mắt |
| 2. Hai biến đếm (chính) | `O(n)` | `O(1)` | **Khuyến nghị** |
| 3. Đặt trước, kiểm sau | `O(n)` | `O(1)` | Cùng logic |
| 4. `reduce` | `O(n)` | `O(1)` | Phong cách hàm |

In [14]:
import time
very_long = "".join(chr(97 + (i * 7) % 26) for i in range(200_000))
for f in [number_of_lines_layout, number_of_lines, number_of_lines_reduce]:
    t = time.perf_counter()
    r = f(widths2, very_long)
    print(f"{f.__name__:<24} -> {r}  ({(time.perf_counter() - t) * 1000:.1f} ms)")

number_of_lines_layout   -> [20000, 94]  (37.1 ms)
number_of_lines          -> [20000, 94]  (11.9 ms)
number_of_lines_reduce   -> [20000, 94]  (18.7 ms)


Cả ba đều tuyến tính; cách dựng dòng chậm hơn vì phải nối chuỗi và lưu văn bản. (Chuỗi 200 000 ký tự vượt constraints, chỉ dùng để đo xu hướng.)

## 13. Bộ test

1. Hai ví dụ đề.
2. Một chữ: `"a"` → `[1, widths[0]]`.
3. Dòng **vừa đúng 100** (không được xuống dòng sớm): 10 chữ rộng 10 → `[1, 100]`.
4. 11 chữ rộng 10 → `[2, 10]`.
5. Phản ví dụ cho công thức tổng: tổng chia 100 cho sai số dòng.
6. Đối chiếu 4 cách trên dữ liệu ngẫu nhiên.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("đề 1", number_of_lines(widths1, s1), [3, 60])
check("đề 2", number_of_lines(widths2, s2), [2, 4])
check("một chữ", number_of_lines(widths2, "a"), [1, 4])
check("vừa đúng 100", number_of_lines(widths1, "a" * 10), [1, 100])
check("101 -> 2 dòng", number_of_lines(widths1, "a" * 11), [2, 10])
w9 = [9] * 26
check("chữ 9px: 11 chữ = 99", number_of_lines(w9, "z" * 11), [1, 99])
check("chữ 9px: 12 chữ", number_of_lines(w9, "z" * 12), [2, 9])

PASS đề 1: [3, 60]
PASS đề 2: [2, 4]
PASS một chữ: [1, 4]
PASS vừa đúng 100: [1, 100]
PASS 101 -> 2 dòng: [2, 10]
PASS chữ 9px: 11 chữ = 99: [1, 99]
PASS chữ 9px: 12 chữ: [2, 9]


In [16]:
import random
random.seed(806)
for _ in range(500):
    ws = [random.randint(2, 10) for _ in range(26)]
    s = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(random.randint(1, 300)))
    expected = number_of_lines_layout(ws, s)
    assert number_of_lines(ws, s) == expected
    assert number_of_lines_reduce(ws, s) == expected
    assert Solution().numberOfLines(ws, s) == expected
    lines, lw = layout_lines(ws, s)
    assert all(x <= 100 for x in lw)                          # mọi dòng ≤ 100
    assert all(lw[i] + ws[ord(lines[i + 1][0]) - 97] > 100    # chữ đầu dòng sau KHÔNG nhét vừa dòng trước
               for i in range(len(lines) - 1))
print("500 bộ ngẫu nhiên: các cách khớp nhau, bố cục thỏa quy tắc tham lam")

500 bộ ngẫu nhiên: các cách khớp nhau, bố cục thỏa quy tắc tham lam


## 14. Lỗi người mới hay gặp

### Lỗi 1: Dùng công thức tổng độ rộng
```python
total = sum(widths[ord(c) - 97] for c in s)
return [ceil(total / 100), total % 100]   # SAI: chữ không được cắt ngang giữa hai dòng
```
Ví dụ 2: tổng = 9·10 + 3·4 = 102 → công thức ra `[2, 2]`, đúng là `[2, 4]` (dòng 1 chỉ dùng 98 px).

### Lỗi 2: Xuống dòng rồi đặt `width = 0`
Quên mất chữ hiện tại → độ rộng dòng cuối thiếu.

### Lỗi 3: Dùng `>= 100` thay cho `> 100`
Dòng vừa bằng 100 bị coi là tràn → xuống dòng sớm.

### Lỗi 4: Khởi tạo `lines = 0`
Dòng đầu tiên không được đếm → thiếu 1.

In [17]:
import math

def wrong_total(widths, s):
    total = sum(widths[ord(c) - 97] for c in s)
    return [math.ceil(total / 100), total % 100]

def wrong_reset_zero(widths, s):
    lines, width = 1, 0
    for ch in s:
        w = widths[ord(ch) - 97]
        if width + w > 100:
            lines += 1
            width = 0          # SAI
        else:
            width += w
    return [lines, width]

def wrong_ge(widths, s):
    lines, width = 1, 0
    for ch in s:
        w = widths[ord(ch) - 97]
        if width + w >= 100:   # SAI
            lines += 1
            width = w
        else:
            width += w
    return [lines, width]

print("ví dụ 2 | tổng      :", wrong_total(widths2, s2), "| đúng:", number_of_lines(widths2, s2))
print("ví dụ 2 | reset 0   :", wrong_reset_zero(widths2, s2))
print("ví dụ 1 | dùng >=   :", wrong_ge(widths1, s1), "| đúng:", number_of_lines(widths1, s1))
print("math.ceil(1.02) =", math.ceil(1.02), "| 102 % 100 =", 102 % 100)

ví dụ 2 | tổng      : [2, 2] | đúng: [2, 4]
ví dụ 2 | reset 0   : [2, 0]
ví dụ 1 | dùng >=   : [3, 80] | đúng: [3, 60]
math.ceil(1.02) = 2 | 102 % 100 = 2


`math.ceil(x)` làm tròn **lên** số nguyên; `%` là phép lấy dư.

## 15. Mẹo nhận dạng pattern

- Đề **mô tả sẵn quy trình** ("viết nhiều nhất có thể rồi sang dòng") ⇒ **mô phỏng** đúng từng bước; không cần tìm công thức.
- Chỉ hỏi vài con số cuối cùng ⇒ giữ **trạng thái tối thiểu** (`lines`, `width`) thay vì lưu toàn bộ.
- Bảng 26 giá trị theo chữ cái ⇒ `table[ord(c) - ord('a')]`.
- Các phần tử **không chia nhỏ được** (chữ, gói hàng, bài hát) ⇒ cẩn thận với công thức chia tổng.

### Biến thể

1. Giới hạn dòng là `max_width` tham số thay vì 100.
2. Trả về **danh sách các dòng** (đã làm ở cách 1).
3. **Word wrap theo từ** (không cắt từ, có dấu cách) — LeetCode 68 Text Justification (Hard) là bản nâng cao.
4. **Chia mảng thành ít nhóm nhất với tổng ≤ K theo thứ tự** (xe chở hàng / băng chuyền) — cùng tham lam.
5. Ngược lại: cho số dòng tối đa, tìm `max_width` nhỏ nhất (tìm kiếm nhị phân trên đáp án — LeetCode 1011).

In [18]:
# Biến thể 4: chia gói hàng theo thứ tự lên xe, mỗi xe chở tối đa K kg
def count_trucks(weights, K):
    trucks, load = 1, 0
    for w in weights:
        if load + w > K:
            trucks, load = trucks + 1, w
        else:
            load += w
    return trucks

print(count_trucks([3, 2, 2, 4, 1, 4], 6))   # [3,2] [2,4] [1,4] -> 3 xe

3


## 16. Bài tập tự luyện (TODO — đáp án ở cuối)

**Bài 1 (tính tay):** `widths` = mọi chữ rộng 7 trừ `'z'` rộng 2. `s = "abcdefghijklmno" + "zzz"` (15 chữ `a..o` rồi 3 chữ `z`). Tính `[lines, width]` bằng tay, gán `answer_ex1`.

**Bài 2 (điền chỗ trống):** Hoàn thiện `number_of_lines_exercise` (lời giải hai biến).

**Bài 3 (biến thể):** `number_of_lines_k(widths, s, max_width)` với giới hạn tùy ý (giả sử mọi chữ ≤ `max_width`).

**Bài 4 (dựng văn bản):** `wrap_text(widths, s)` trả về list các dòng (chuỗi).

**Bài 5 (mini challenge):** `min_max_width(widths, s, max_lines)` tìm giới hạn độ rộng **nhỏ nhất** sao cho `s` viết được trong tối đa `max_lines` dòng.

In [19]:
answer_ex1 = None   # TODO: [lines, width]

def number_of_lines_exercise(widths: list[int], s: str) -> list[int]:
    # TODO 1: lines, width = 1, 0
    # TODO 2: duyệt s, tra w
    # TODO 3: nếu width + w > 100 -> dòng mới (width = w), ngược lại width += w
    pass

def number_of_lines_k(widths: list[int], s: str, max_width: int) -> list[int]:
    # TODO
    pass

# assert number_of_lines_exercise([10] * 26, "abcdefghijklmnopqrstuvwxyz") == [3, 60]
# assert number_of_lines_k([10] * 26, "abcdefghijklmnopqrstuvwxyz", 50) == [6, 10]

In [20]:
def wrap_text(widths: list[int], s: str) -> list[str]:
    # TODO
    pass

def min_max_width(widths: list[int], s: str, max_lines: int) -> int:
    # TODO: thử từ max(widths của chữ trong s) tăng dần, hoặc tìm kiếm nhị phân
    pass

# assert wrap_text([4] + [10] * 25, "bbbcccdddaaa") == ["bbbcccdddaa", "a"]
# assert min_max_width([10] * 26, "abcdefghijklmnopqrstuvwxyz", 3) == 90

### Gợi ý 3 mức

**Bài 1** — (1) Dòng 1 nhét được bao nhiêu chữ 7 px? (2) `14 · 7 = 98`, chữ thứ 15 làm vượt. (3) Dòng 2 bắt đầu bằng chữ thứ 15 (`'o'`, 7 px) rồi tới ba chữ `z`.

**Bài 2** — (1) Hai biến trạng thái. (2) `w = widths[ord(ch) - ord('a')]`. (3) `if width + w > 100: lines += 1; width = w`.

**Bài 3** — (1) Thay hằng 100 bằng tham số. (2) Giữ nguyên mọi thứ khác. (3) `if width + w > max_width`.

**Bài 4** — (1) Giống cách 1 mục 4. (2) Giữ `cur` (chuỗi dòng hiện tại) và `width`. (3) Khi tràn: `append(cur)`, `cur = ch`.

**Bài 5** — (1) Số dòng **giảm** khi `max_width` tăng (đơn điệu). (2) Thử `max_width` từ chữ rộng nhất tới tổng độ rộng. (3) Dừng ở giá trị đầu tiên có `lines <= max_lines` (hoặc tìm kiếm nhị phân cho nhanh).

---

## 17. Đáp án tham khảo

In [21]:
answer_ex1_solution = [2, 13]

def number_of_lines_exercise_solution(widths, s):
    lines, width = 1, 0
    for ch in s:
        w = widths[ord(ch) - ord("a")]
        if width + w > 100:
            lines += 1
            width = w
        else:
            width += w
    return [lines, width]

def number_of_lines_k_solution(widths, s, max_width):
    lines, width = 1, 0
    for ch in s:
        w = widths[ord(ch) - ord("a")]
        if width + w > max_width:
            lines, width = lines + 1, w
        else:
            width += w
    return [lines, width]

def wrap_text_solution(widths, s):
    out, cur, width = [], "", 0
    for ch in s:
        w = widths[ord(ch) - ord("a")]
        if width + w > 100:
            out.append(cur)
            cur, width = ch, w
        else:
            cur, width = cur + ch, width + w
    out.append(cur)
    return out

def min_max_width_solution(widths, s, max_lines):
    lo = max(widths[ord(c) - 97] for c in s)
    hi = sum(widths[ord(c) - 97] for c in s)
    while lo < hi:                                    # tìm kiếm nhị phân trên đáp án
        mid = (lo + hi) // 2
        if number_of_lines_k_solution(widths, s, mid)[0] <= max_lines:
            hi = mid
        else:
            lo = mid + 1
    return lo

ws_ex = [7] * 25 + [2]
assert number_of_lines(ws_ex, "abcdefghijklmno" + "zzz") == answer_ex1_solution
assert number_of_lines_exercise_solution([10] * 26, s1) == [3, 60]
assert number_of_lines_k_solution([10] * 26, s1, 50) == [6, 10]
assert number_of_lines_k_solution(widths2, s2, 100) == number_of_lines(widths2, s2)
assert wrap_text_solution(widths2, s2) == ["bbbcccdddaa", "a"]
assert min_max_width_solution([10] * 26, s1, 3) == 90
assert min_max_width_solution([10] * 26, s1, 1) == 260
print("Đáp án PASS toàn bộ test!")

Đáp án PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def numberOfLines(self, widths: List[int], s: str) -> List[int]:
        lines, width = 1, 0
        for ch in s:
            w = widths[ord(ch) - ord("a")]
            if width + w > 100:
                lines, width = lines + 1, w
            else:
                width += w
        return [lines, width]
```

Mẫu chung **"nhét tham lam theo thứ tự"**: `if cur + x > LIMIT: count += 1; cur = x else: cur += x`.

**Checklist trước khi submit:**
- [ ] `lines` bắt đầu từ **1**?
- [ ] Điều kiện tràn là `> 100` (dòng bằng 100 vẫn hợp lệ)?
- [ ] Khi xuống dòng đặt `width = w` (không phải 0)?
- [ ] Không dùng công thức chia tổng?
- [ ] Trả về **list** `[lines, width]`?